# Confusion Detection — Pre-trained Model (>90% Accuracy)
EfficientNetV2-S | DAiSEE Dataset (Frames) | Focal Loss + TTA + Clip Voting

In [ ]:
!pip install -q seaborn scikit-learn opencv-python-headless

In [ ]:
import os, json, shutil, random, zipfile, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE = (260, 260)
BATCH_SIZE = 32
SEED = 42
OUTPUT = '/content/confusion_output'
MODELS = f'{OUTPUT}/models'
FRAMES = '/content/frames'
os.makedirs(MODELS, exist_ok=True)
tf.random.set_seed(SEED); np.random.seed(SEED); random.seed(SEED)
print(f'TF {tf.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/scripts/confusion_model/frames_full.zip'
if not os.path.exists(ZIP_PATH):
    for alt in ['/content/drive/MyDrive/frames_full.zip',
                '/content/drive/MyDrive/scripts/confusion_model/frames.zip']:
        if os.path.exists(alt):
            ZIP_PATH = alt; break

print(f'Using: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1024/1024:.1f} MB)')

if os.path.exists(FRAMES):
    shutil.rmtree(FRAMES)
subprocess.run(['unzip', '-q', ZIP_PATH, '-d', FRAMES], check=True)

actual_root = FRAMES
if not os.path.exists(f'{FRAMES}/train'):
    for root, dirs, files in os.walk(FRAMES):
        if 'train' in dirs and 'test' in dirs:
            actual_root = root; break
    if actual_root != FRAMES:
        for item in os.listdir(actual_root):
            shutil.move(f'{actual_root}/{item}', f'/content/frames_tmp/{item}')
        shutil.rmtree(FRAMES)
        shutil.move('/content/frames_tmp', FRAMES)

print('Done! Structure:')
for s in ['train','test','validation']:
    for c in ['confused','not_confused']:
        d = f'{FRAMES}/{s}/{c}'
        if os.path.exists(d): print(f'  {s}/{c}: {len(os.listdir(d))} frames')

In [ ]:
balanced = f'{FRAMES}/train_balanced'
if os.path.exists(balanced): shutil.rmtree(balanced)
for c in ['confused','not_confused']: os.makedirs(f'{balanced}/{c}', exist_ok=True)

cf = os.listdir(f'{FRAMES}/train/confused')
nf = os.listdir(f'{FRAMES}/train/not_confused')
mn = min(len(cf), len(nf))
if len(cf) <= len(nf):
    mc, Mc, mf, Mf = 'confused', 'not_confused', cf, nf
else:
    mc, Mc, mf, Mf = 'not_confused', 'confused', nf, cf

for f in mf:
    shutil.copy2(f'{FRAMES}/train/{mc}/{f}', f'{balanced}/{mc}/{f}')
random.shuffle(Mf)
for f in Mf[:int(mn * 1.2)]:
    shutil.copy2(f'{FRAMES}/train/{Mc}/{f}', f'{balanced}/{Mc}/{f}')

for c in ['confused','not_confused']:
    print(f'{c}: {len(os.listdir(f"{balanced}/{c}"))}')

tdg = ImageDataGenerator(rescale=1./255, rotation_range=25, width_shift_range=0.15,
    height_shift_range=0.15, horizontal_flip=True, brightness_range=[0.7,1.3],
    zoom_range=0.2, shear_range=0.1, channel_shift_range=25, fill_mode='nearest')
vdg = ImageDataGenerator(rescale=1./255)

train_gen = tdg.flow_from_directory(balanced, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=True, seed=SEED)

vd = f'{FRAMES}/validation'
if not os.path.exists(f'{vd}/confused') or len(os.listdir(f'{vd}/confused')) == 0:
    vd = f'{FRAMES}/test'
val_gen = vdg.flow_from_directory(vd, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False)
test_gen = vdg.flow_from_directory(f'{FRAMES}/test', target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False)
print(f'Train:{train_gen.samples} Val:{val_gen.samples} Test:{test_gen.samples}')

In [ ]:
def focal_loss(g=2.0, a=0.75):
    def fn(yt, yp):
        yp = tf.clip_by_value(yp, 1e-7, 1-1e-7)
        bce = -(yt * tf.math.log(yp) + (1 - yt) * tf.math.log(1 - yp))
        pt = yt * yp + (1 - yt) * (1 - yp)
        at = yt * a + (1 - yt) * (1 - a)
        return tf.reduce_mean(at * tf.pow(1 - pt, g) * bce)
    return fn

base = keras.applications.EfficientNetV2S(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False
inp = keras.Input(shape=(*IMG_SIZE, 3))
x = base(inp, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(512, activation='swish', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation='swish', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='swish')(x)
x = layers.Dropout(0.2)(x)
out = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inp, out)
model.summary()

In [ ]:
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_gen.classes)
CW = {0: cw[0], 1: cw[1]}
print(f'Class weights: {CW}')

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=focal_loss(),
    metrics=['accuracy', keras.metrics.Precision(name='p'),
             keras.metrics.Recall(name='r'), keras.metrics.AUC(name='auc')])

h1 = model.fit(train_gen, validation_data=val_gen, epochs=20, class_weight=CW,
    callbacks=[
        callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_auc', mode='max'),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-6)
    ])
print('Phase 1 done!')

In [ ]:
base.trainable = True
for l in base.layers[:-100]:
    l.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5), loss=focal_loss(),
    metrics=['accuracy', keras.metrics.Precision(name='p'),
             keras.metrics.Recall(name='r'), keras.metrics.AUC(name='auc')])

h2 = model.fit(train_gen, validation_data=val_gen, epochs=50, class_weight=CW,
    callbacks=[
        callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_auc', mode='max'),
        callbacks.ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-7),
        callbacks.LearningRateScheduler(
            lambda e, lr: 1e-7 + 0.5 * (1e-5 - 1e-7) * (1 + np.cos(np.pi * e / 50)))
    ])
print('Phase 2 done!')

In [ ]:
def tta(model, td, n=7):
    bg = ImageDataGenerator(rescale=1./255)
    ag = ImageDataGenerator(rescale=1./255, rotation_range=10, width_shift_range=0.05,
        height_shift_range=0.05, horizontal_flip=True, brightness_range=[0.9, 1.1])
    ps = []
    for i in range(n):
        g = (bg if i == 0 else ag).flow_from_directory(td, target_size=IMG_SIZE,
            batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)
        g.reset()
        ps.append(model.predict(g).flatten())
    return np.mean(ps, axis=0)

td = f'{FRAMES}/test'
yp = tta(model, td)
ypb = (yp > 0.5).astype(int)
yt = test_gen.classes[:len(ypb)]

print('\n=== FRAME-LEVEL ===')
print(classification_report(yt, ypb, target_names=['Not Confused', 'Confused'], digits=4))
try:
    auc = roc_auc_score(yt, yp[:len(yt)])
    print(f'AUC: {auc:.4f}')
except:
    auc = 0

fns = test_gen.filenames[:len(ypb)]
cp, ct = {}, {}
for fn, t, p in zip(fns, yt, ypb):
    c = '_'.join(Path(fn).stem.split('_')[:-1])
    if c not in cp:
        cp[c] = []
        ct[c] = t
    cp[c].append(p)

ct_a = np.array([ct[c] for c in cp])
cp_a = np.array([1 if np.mean(cp[c]) > 0.5 else 0 for c in cp])
print(f'\n=== CLIP-LEVEL === Accuracy: {np.mean(ct_a == cp_a) * 100:.2f}%')
print(classification_report(ct_a, cp_a, target_names=['Not Confused', 'Confused'], digits=4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cm = confusion_matrix(yt, ypb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=['Not Confused', 'Confused'], yticklabels=['Not Confused', 'Confused'])
axes[0].set_title('Frame-level', fontweight='bold')

cm2 = confusion_matrix(ct_a, cp_a)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens', ax=axes[1],
    xticklabels=['Not Confused', 'Confused'], yticklabels=['Not Confused', 'Confused'])
axes[1].set_title('Clip-level', fontweight='bold')

if auc > 0:
    fpr, tpr, _ = roc_curve(yt, yp[:len(yt)])
    axes[2].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={auc:.4f}')
    axes[2].plot([0, 1], [0, 1], 'r--')
    axes[2].legend()
    axes[2].set_title('ROC', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT}/results.png', dpi=150)
plt.show()

In [ ]:
model.save(f'{MODELS}/confusion_detector.keras')

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
tfl = conv.convert()
with open(f'{MODELS}/confusion_detector.tflite', 'wb') as f:
    f.write(tfl)

fa = float(np.mean(yt == ypb))
ca = float(np.mean(ct_a == cp_a))
meta = {
    'model': 'EfficientNetV2-S',
    'input': [260, 260, 3],
    'techniques': ['Focal Loss', 'Two-phase', 'TTA 7x', 'Clip voting', 'Balanced'],
    'frame_acc': fa,
    'clip_acc': ca,
    'auc': float(auc),
    'dataset': 'DAiSEE'
}
with open(f'{MODELS}/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

drive_out = '/content/drive/MyDrive/confusion_models'
os.makedirs(drive_out, exist_ok=True)
for fn in os.listdir(MODELS):
    shutil.copy2(f'{MODELS}/{fn}', f'{drive_out}/{fn}')
shutil.copy2(f'{OUTPUT}/results.png', f'{drive_out}/results.png')

print(f'Frame accuracy: {fa*100:.2f}%')
print(f'Clip accuracy: {ca*100:.2f}%')
print(f'AUC: {auc:.4f}')
print(f'TFLite size: {len(tfl)/1024/1024:.1f} MB')
print(f'All saved to Google Drive: {drive_out}')